## Logging to HuggingFce


In [ ]:
#!pip install unsloth trl datasets torch bitsandbytes huggingface_hub loguru

# kaggle version
!pip install -q unsloth trl datasets bitsandbytes huggingface_hub loguru -q



In [ ]:
# login for kaggle




from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")




In [ ]:
!pip install -q huggingface_hub 

from huggingface_hub import login
login(token=hf_token)

## Phase 1 - Training

In [ ]:
# ─────────────────────────────────────────────
# 0. MUST BE FIRST (before torch/unsloth)
# ─────────────────────────────────────────────
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# Patch out the broken warmup allocator in Transformers 5.5.0
import transformers.modeling_utils as _mu
_mu.caching_allocator_warmup = lambda *a, **kw: None



# REMOVED: os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Keeping both GPUs visible so the 14B model can spread across 2x T4s (~29 GB total)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# IMPORTANT: do NOT disable xformers unless debugging
# os.environ["XFORMERS_DISABLED"] = "1"

# ─────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────
import json
import gc
import torch

from pathlib import Path
from loguru import logger
from datasets import Dataset

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig


# ─────────────────────────────────────────────
# 2. LOGGING
# ─────────────────────────────────────────────
Path("logs").mkdir(exist_ok=True)

logger.add(
    "logs/phase1_training.log",
    rotation="100 MB",
    retention="7 days",
    level="DEBUG",
)


# ─────────────────────────────────────────────
# 3. CONFIG
# ─────────────────────────────────────────────
MODEL_NAME = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"


DATA_PATH = "/kaggle/input/datasets/fcyber/dataset/sec_finetune_v20_clean_fixed.jsonl"

OUTPUT_DIR = "FinReasoner-qwen2.5-14b-phase1"

HF_REPO = "fcyber/FinReasoner-qwen2.5-14b-instruct-phase1"

MAX_SEQ_LEN = 1024

LORA_RANK = 16

BATCH_SIZE = 1
GRAD_ACCUM = 4
EPOCHS = 2
LR = 1e-4

SEED = 42
PRIVATE = True


# ─────────────────────────────────────────────
# 4. CLEAN GPU BEFORE LOAD
# ─────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


# ─────────────────────────────────────────────
# 5. LOAD MODEL
# FIX: device_map="auto" spreads layers across both T4s instead of
#      cramming everything onto cuda:0 (14.5 GB is not enough alone).
# ─────────────────────────────────────────────
logger.info("Loading base model")


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,

    dtype=torch.float16,
    load_in_4bit=True,

    device_map="auto",
    max_memory={0: "9GiB", 1: "9GiB"},   # ✅ ~5.5GB headroom each for activations
)
# ─────────────────────────────────────────────
# 6. APPLY LoRA (rsLoRA)
# ─────────────────────────────────────────────
logger.info("Applying LoRA")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,

    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    use_rslora=True,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()


# ─────────────────────────────────────────────
# 7. LOAD DATA
# ─────────────────────────────────────────────
logger.info("Loading dataset")

records = []
with open(DATA_PATH, "r") as f:
    for line in f:
        records.append(json.loads(line))


# ─────────────────────────────────────────────
# 8. FORMAT DATA
# ─────────────────────────────────────────────
def format_example(row):
    ip = row["instruction_pair"]

    messages = [
        {"role": "system",    "content": ip["instruction"]},
        {"role": "user",      "content": ip["input"]},
        {"role": "assistant", "content":
            ip["output"] if isinstance(ip["output"], str)
            else json.dumps(ip["output"], ensure_ascii=False)
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


formatted = [format_example(r) for r in records]
dataset   = Dataset.from_list(formatted)

split = dataset.train_test_split(test_size=0.05, seed=SEED)

train_ds = split["train"]
eval_ds  = split["test"]


# ─────────────────────────────────────────────
# 9. TRAINER
# ─────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=train_ds,
    eval_dataset=eval_ds,

    args=SFTConfig(
        output_dir=OUTPUT_DIR,

        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,

        learning_rate=LR,
        lr_scheduler_type="cosine",

        warmup_steps=100,

        fp16=True,
        bf16=False,

        logging_steps=25,

        eval_strategy="epoch",
        save_strategy="epoch",

        save_total_limit=2,
        load_best_model_at_end=True,

        max_grad_norm=1.0,
        dataset_text_field="text",

        max_seq_length=MAX_SEQ_LEN,

        packing=True,              # important speed boost

        dataloader_pin_memory=False,  # ✅ FIX: avoids pinning issues on multi-GPU Kaggle

        report_to="none",
        seed=SEED,
    ),
)


# ─────────────────────────────────────────────
# 10. TRAIN
# ─────────────────────────────────────────────
logger.info("Starting training")

trainer.train()

logger.success("Training done")


# ─────────────────────────────────────────────
# 11. SAVE
# ─────────────────────────────────────────────
logger.info("Saving model")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


# ─────────────────────────────────────────────
# 12. PUSH TO HUB
# ─────────────────────────────────────────────
logger.info("Pushing to HF")

model.push_to_hub(HF_REPO, private=PRIVATE)
tokenizer.push_to_hub(HF_REPO, private=PRIVATE)

logger.success("Done")

## Phase 2

In [ ]:
import json
import torch
import re
from pathlib import Path
from loguru import logger
from tqdm import tqdm
from unsloth import FastLanguageModel
from datasets import Dataset

# ── LOGGING SETUP ────────────────────────────────────────
logger.add(
    "logs/phase2_evaluation.log",
    rotation  = "100 MB",
    retention = "7 days",
    level     = "DEBUG",
    format    = "{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}",
)

# ── CONFIG ──────────────────────────────────────────────
MODEL_PATH     = "FinReasoner-qwen2.5-14b-instruct-phase1"
EVAL_PATH      = "eval_set.jsonl"
OUTPUT_FILE    = "phase2_results.jsonl"
MAX_SEQ_LEN    = 4096
MAX_NEW_TOKENS = 1024
# ────────────────────────────────────────────────────────


# ── LOAD MODEL ───────────────────────────────────────────
logger.info("Loading Phase 1 model from: {}", MODEL_PATH)
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_PATH,
        max_seq_length = MAX_SEQ_LEN,
        dtype          = torch.bfloat16,
        load_in_4bit   = True,
    )
    FastLanguageModel.for_inference(model)
    logger.success("Model loaded for inference")
except Exception as e:
    logger.error("Failed to load model: {}", e)
    raise


# ── LOAD EVAL DATA ───────────────────────────────────────
logger.info("Loading eval data from: {}", EVAL_PATH)
try:
    eval_data = []
    with open(EVAL_PATH) as f:
        for line in f:
            eval_data.append(json.loads(line))
    logger.success("Loaded {} eval records", len(eval_data))
except Exception as e:
    logger.error("Failed to load eval data: {}", e)
    raise


# ── GENERATE ─────────────────────────────────────────────
def generate(instruction, input_text):
    try:
        messages = [
            {"role": "system", "content": instruction},
            {"role": "user",   "content": input_text},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize              = False,
            add_generation_prompt = True,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = MAX_NEW_TOKENS,
                temperature    = 0.1,
                do_sample      = True,
                pad_token_id   = tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        return decoded.strip()
    except Exception as e:
        logger.error("Generation failed: {}", e)
        return ""


# ── BENCHMARK FUNCTIONS ──────────────────────────────────
UNCERTAINTY_PHRASES = [
    "not disclosed",
    "cannot be determined",
    "evidence is inconclusive",
    "estimated from available disclosures",
    "calculated from reported values",
    "insufficient evidence",
]

CAUSAL_PHRASES = [
    "driven by", "offset by", "primarily due to",
    "resulted from", "attributable to",
]

KEY_METRICS = [
    "revenue", "net income", "ebitda",
    "eps", "fcf", "capex", "margin"
]

def check_hallucination(output, input_text):
    numbers_in_input  = set(re.findall(r'\d+\.?\d*', input_text))
    numbers_in_output = set(re.findall(r'\d+\.?\d*', output))
    invented = {n for n in (numbers_in_output - numbers_in_input)
                if float(n) > 10}
    return {
        "hallucination_flag" : len(invented) > 5,
        "invented_count"     : len(invented),
    }

def check_uncertainty(output):
    found = [p for p in UNCERTAINTY_PHRASES if p in output.lower()]
    return {
        "uncertainty_phrases_found" : len(found),
        "uses_uncertainty_language" : len(found) > 0,
    }

def check_causal_discipline(output):
    found = [p for p in CAUSAL_PHRASES if p in output.lower()]
    return {
        "causal_phrases_count"  : len(found),
        "has_causal_discipline" : len(found) >= 2,
    }

def check_format(output):
    try:
        cleaned = re.sub(r"```json|```", "", output).strip()
        json.loads(cleaned)
        return {"valid_json": True}
    except Exception:
        return {"valid_json": False}

def check_evidence_citation(output):
    cited = [m for m in KEY_METRICS if m in output.lower()]
    return {
        "metrics_cited" : len(cited),
        "evidence_rich" : len(cited) >= 4,
    }

def check_calibration(output, confidence_label):
    hedge_words  = ["approximately", "estimated", "suggests",
                    "indicates", "appears", "likely"]
    hedges_found = sum(1 for w in hedge_words if w in output.lower())
    calibrated   = not (confidence_label == "LOW" and hedges_found == 0)
    return {
        "hedge_words_found" : hedges_found,
        "calibrated"        : calibrated,
    }


# ── MAIN EVAL LOOP ───────────────────────────────────────
logger.info("Starting evaluation loop on {} records", len(eval_data))

results = []
summary = {
    "total"                 : 0,
    "hallucination_flags"   : 0,
    "uses_uncertainty"      : 0,
    "has_causal_discipline" : 0,
    "valid_json"            : 0,
    "evidence_rich"         : 0,
    "calibrated"            : 0,
    "generation_errors"     : 0,
}

for i, row in enumerate(tqdm(eval_data, desc="Evaluating")):
    try:
        ip          = row["instruction_pair"]
        instruction = ip["instruction"]
        input_text  = ip["input"]
        reference   = ip["output"] if isinstance(ip["output"], str) \
                      else json.dumps(ip["output"])
        confidence  = row.get("meta", {}).get("sanity_confidence", "HIGH")
        ticker      = row.get("meta", {}).get("ticker", "unknown")

        prediction  = generate(instruction, input_text)

        if not prediction:
            logger.warning("Empty generation for record {} ticker={}", i, ticker)
            summary["generation_errors"] += 1
            continue

        h   = check_hallucination(prediction, input_text)
        u   = check_uncertainty(prediction)
        c   = check_causal_discipline(prediction)
        f   = check_format(prediction)
        e   = check_evidence_citation(prediction)
        cal = check_calibration(prediction, confidence)

        # log individual warnings
        if h["hallucination_flag"]:
            logger.warning("Hallucination detected | ticker={} | invented_numbers={}",
                           ticker, h["invented_count"])
        if not f["valid_json"]:
            logger.warning("Invalid JSON output | ticker={}", ticker)
        if not cal["calibrated"]:
            logger.warning("Calibration failure | ticker={} | confidence={}",
                           ticker, confidence)

        record = {
            "id"        : row.get("id", ""),
            "ticker"    : ticker,
            "form_type" : row.get("meta", {}).get("form_type", ""),
            "prediction": prediction,
            "reference" : reference,
            **h, **u, **c, **f, **e, **cal,
        }
        results.append(record)

        summary["total"]                 += 1
        summary["hallucination_flags"]   += int(h["hallucination_flag"])
        summary["uses_uncertainty"]      += int(u["uses_uncertainty_language"])
        summary["has_causal_discipline"] += int(c["has_causal_discipline"])
        summary["valid_json"]            += int(f["valid_json"])
        summary["evidence_rich"]         += int(e["evidence_rich"])
        summary["calibrated"]            += int(cal["calibrated"])

    except Exception as e:
        logger.error("Unexpected error at record {} | error={}", i, e)
        continue


# ── SAVE RESULTS ─────────────────────────────────────────
logger.info("Saving results to: {}", OUTPUT_FILE)
try:
    with open(OUTPUT_FILE, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
    logger.success("Results saved: {} records", len(results))
except Exception as e:
    logger.error("Failed to save results: {}", e)
    raise


# ── PRINT + LOG SUMMARY ──────────────────────────────────
n = summary["total"]

summary_text = f"""
══════════════ PHASE 2 RESULTS ══════════════
Total evaluated          : {n}
Generation errors        : {summary['generation_errors']}
Hallucination rate       : {summary['hallucination_flags']/n*100:.1f}%  ← want < 5%
Uncertainty language     : {summary['uses_uncertainty']/n*100:.1f}%  ← want > 60%
Causal discipline        : {summary['has_causal_discipline']/n*100:.1f}%  ← want > 80%
Valid JSON format        : {summary['valid_json']/n*100:.1f}%  ← want > 95%
Evidence rich outputs    : {summary['evidence_rich']/n*100:.1f}%  ← want > 80%
Calibrated confidence    : {summary['calibrated']/n*100:.1f}%  ← want > 85%
═════════════════════════════════════════════
"""

print(summary_text)
logger.info(summary_text)


# ── DECISION GUIDANCE ────────────────────────────────────
hal_rate = summary['hallucination_flags'] / n * 100
cal_rate = summary['calibrated'] / n * 100
unc_rate = summary['uses_uncertainty'] / n * 100

if hal_rate < 5 and cal_rate > 85 and unc_rate > 60:
    logger.success("All targets met. Skip Phase 3. Push Phase 1 model to production.")
else:
    if hal_rate >= 5:
        logger.warning("Hallucination too high ({:.1f}%). Add refusal examples → retrain.", hal_rate)
    if cal_rate <= 85:
        logger.warning("Calibration too low ({:.1f}%). Proceed to Phase 3 DPO.", cal_rate)
    if unc_rate <= 60:
        logger.warning("Uncertainty language too low ({:.1f}%). Add not-disclosed examples → retrain.", unc_rate)

logger.info("Phase 2 complete")

# PHASE 3

In [ ]:
import json
import torch
from pathlib import Path
from loguru import logger
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from huggingface_hub import login

# ── LOGGING SETUP ────────────────────────────────────────
logger.add(
    "logs/phase3_dpo.log",
    rotation  = "100 MB",
    retention = "7 days",
    level     = "DEBUG",
    format    = "{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}",
)

# ── CONFIG ──────────────────────────────────────────────
MODEL_PATH      = "FinReasoner-qwen2.5-14b-instruct-phase1"
DATA_PATH       = "dataset_score80plus.jsonl"
PREF_PATH       = "preference_pairs.jsonl"
OUTPUT_DIR      = "FinReasoner-qwen2.5-14b-instruct"
HF_REPO_PHASE3  = "fcyber/FinReasoner-qwen2.5-14b-instruct-phase3"
HF_REPO_FINAL   = "fcyber/FinReasoner-qwen2.5-14b-instruct"
MAX_SEQ_LEN     = 4096
LORA_RANK       = 16
EPOCHS          = 1
LR              = 5e-5
BETA            = 0.1
PRIVATE         = True
# ────────────────────────────────────────────────────────


# ── LOGIN ────────────────────────────────────────────────
logger.info("Logging into HuggingFace")
try:
    login()
    logger.success("HuggingFace login successful")
except Exception as e:
    logger.error("HuggingFace login failed: {}", e)
    raise


# ── BUILD PREFERENCE PAIRS ───────────────────────────────
logger.info("Building preference pairs from: {}", DATA_PATH)

REJECTED_TEMPLATES = [
    "Operating margin appears to be approximately {val}%, consistent with industry norms.",
    "Revenue growth of {val}% indicates strong operational efficiency improvements.",
    "EBITDA margin expanded to {val}%, driven by cost optimization across all segments.",
]

def make_rejected(row):
    try:
        metrics = row.get("metrics", {})
        val     = round(metrics.get("operating_margin_pct", 15.0), 1)
        template = REJECTED_TEMPLATES[hash(row["id"]) % len(REJECTED_TEMPLATES)]
        return template.format(val=val)
    except Exception as e:
        logger.warning("Failed to make rejected for id={} | error={}", 
                       row.get("id", "unknown"), e)
        return "Performance metrics indicate positive trends across all segments."

def make_chosen(row):
    try:
        ip     = row["instruction_pair"]
        output = ip["output"]
        return output if isinstance(output, str) else json.dumps(output)
    except Exception as e:
        logger.warning("Failed to make chosen for id={} | error={}", 
                       row.get("id", "unknown"), e)
        return None

try:
    pairs = []
    skipped = 0
    with open(DATA_PATH) as f:
        for line in f:
            row   = json.loads(line)
            score = row.get("instruction_pair", {}) \
                       .get("metadata", {}) \
                       .get("quality_score", 0)

            if score < 90:
                skipped += 1
                continue

            ip      = row["instruction_pair"]
            prompt  = ip["instruction"] + "\n\n" + ip["input"]
            chosen  = make_chosen(row)

            if not chosen:
                skipped += 1
                continue

            pairs.append({
                "prompt"   : prompt,
                "chosen"   : chosen,
                "rejected" : make_rejected(row),
            })

    logger.success("Preference pairs built: {} | skipped: {}", len(pairs), skipped)

    with open(PREF_PATH, "w") as f:
        for p in pairs:
            f.write(json.dumps(p) + "\n")
    logger.success("Preference pairs saved to: {}", PREF_PATH)

except Exception as e:
    logger.error("Failed to build preference pairs: {}", e)
    raise


# ── LOAD MODEL ───────────────────────────────────────────
logger.info("Loading Phase 1 model from: {}", MODEL_PATH)
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_PATH,
        max_seq_length = MAX_SEQ_LEN,
        dtype          = torch.bfloat16,
        load_in_4bit   = True,
    )
    logger.success("Model loaded successfully")
except Exception as e:
    logger.error("Failed to load model: {}", e)
    raise

logger.info("Applying rsLoRA for DPO | rank={}", LORA_RANK)
try:
    model = FastLanguageModel.get_peft_model(
        model,
        r                          = LORA_RANK,
        lora_alpha                 = LORA_RANK * 2,
        lora_dropout               = 0.05,
        target_modules             = [
            "q_proj", "k_proj", "v_proj",
            "o_proj", "gate_proj",
            "up_proj", "down_proj"
        ],
        use_rslora                 = True,
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
    )
    logger.success("rsLoRA applied")
except Exception as e:
    logger.error("Failed to apply rsLoRA: {}", e)
    raise


# ── LOAD PREFERENCE DATA ─────────────────────────────────
logger.info("Loading preference pairs from: {}", PREF_PATH)
try:
    raw      = []
    with open(PREF_PATH) as f:
        for line in f:
            raw.append(json.loads(line))
    dataset  = Dataset.from_list(raw)
    split    = dataset.train_test_split(test_size=0.05, seed=42)
    train_ds = split["train"]
    eval_ds  = split["test"]
    logger.success("DPO Train: {} | Eval: {}", len(train_ds), len(eval_ds))
except Exception as e:
    logger.error("Failed to load preference data: {}", e)
    raise


# ── DPO TRAINER ──────────────────────────────────────────
logger.info("Initializing DPO Trainer | beta={} | lr={}", BETA, LR)
try:
    trainer = DPOTrainer(
        model         = model,
        ref_model     = None,
        tokenizer     = tokenizer,
        train_dataset = train_ds,
        eval_dataset  = eval_ds,
        args = DPOConfig(
            output_dir                  = OUTPUT_DIR,
            num_train_epochs            = EPOCHS,
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 8,
            warmup_ratio                = 0.05,
            learning_rate               = LR,
            bf16                        = True,
            logging_steps               = 25,
            eval_strategy               = "steps",
            eval_steps                  = 100,
            save_strategy               = "steps",
            save_steps                  = 100,
            save_total_limit            = 2,
            load_best_model_at_end      = True,
            max_grad_norm               = 1.0,
            beta                        = BETA,
            max_length                  = MAX_SEQ_LEN,
            max_prompt_length           = 2048,
            report_to                   = "none",
        ),
    )
    logger.success("DPO Trainer initialized")
except Exception as e:
    logger.error("Failed to initialize DPO Trainer: {}", e)
    raise


# ── TRAIN ────────────────────────────────────────────────
logger.info("Starting Phase 3 DPO training")
try:
    trainer.train()
    logger.success("DPO training completed")
except Exception as e:
    logger.error("DPO training failed: {}", e)
    raise


# ── SAVE LOCALLY ─────────────────────────────────────────
logger.info("Saving final model locally to: {}", OUTPUT_DIR)
try:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    logger.success("Final model saved locally")
except Exception as e:
    logger.error("Failed to save model locally: {}", e)
    raise


# ── PUSH PHASE 3 CHECKPOINT ──────────────────────────────
logger.info("Pushing Phase 3 checkpoint to: {}", HF_REPO_PHASE3)
try:
    model.push_to_hub(HF_REPO_PHASE3, private=PRIVATE)
    tokenizer.push_to_hub(HF_REPO_PHASE3, private=PRIVATE)
    logger.success("Phase 3 checkpoint pushed to: https://huggingface.co/{}", HF_REPO_PHASE3)
except Exception as e:
    logger.error("Failed to push Phase 3 checkpoint: {}", e)
    raise


# ── PUSH FINAL PRODUCTION MODEL ──────────────────────────
logger.info("Pushing FINAL production model to: {}", HF_REPO_FINAL)
try:
    model.push_to_hub(HF_REPO_FINAL, private=PRIVATE)
    tokenizer.push_to_hub(HF_REPO_FINAL, private=PRIVATE)
    logger.success("FINAL model pushed to: https://huggingface.co/{}", HF_REPO_FINAL)
except Exception as e:
    logger.error("Failed to push final model: {}", e)
    raise

logger.info("Phase 3 complete")

In [ ]:
!zip -r logs.zip logs/